# Sesión 08 - Modelos continuos y teorema del límite central

Objetivo: modelar variables continuas, estimar parámetros y observar el comportamiento de promedios muestrales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(51)
pd.set_option("display.precision", 4)


## 1. Comparación de distribuciones continuas


In [ ]:
xs = np.linspace(0, 20, 500)
distribuciones = {
    "Exponencial": stats.expon(scale=4),
    "Gamma": stats.gamma(a=3, scale=1.5),
    "Lognormal": stats.lognorm(s=0.45, scale=np.exp(2.0)),
    "Normal truncada visual": stats.norm(loc=10, scale=2),
}

fig, ax = plt.subplots(figsize=(9, 4))
for nombre, dist in distribuciones.items():
    ax.plot(xs, dist.pdf(xs), label=nombre)
ax.set_title("PDF de modelos continuos frecuentes")
ax.set_xlabel("x")
ax.set_ylabel("densidad")
ax.legend()
plt.show()


## 2. Ajuste simple de distribución

Simulamos tiempos positivos con cola derecha y comparamos ajuste exponencial vs lognormal.


### Lectura matemática

- **Distribuciones candidatas:** Exponencial, Gamma y Lognormal para variables positivas.
- **Parámetros estimados:** tasa, forma, escala o parámetros lognormales.
- **Supuesto que puede fallar:** soporte incorrecto, mezcla de poblaciones o colas mal modeladas.
- **Diagnóstico:** KS, CDF empírica, QQ plot y probabilidades de cola.


In [ ]:
tiempos = rng.lognormal(mean=2.0, sigma=0.5, size=1_000)

loc_exp, scale_exp = stats.expon.fit(tiempos, floc=0)
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(tiempos, floc=0)

ks_exp = stats.kstest(tiempos, "expon", args=(loc_exp, scale_exp))
ks_ln = stats.kstest(tiempos, "lognorm", args=(shape_ln, loc_ln, scale_ln))

resultados = pd.DataFrame(
    {
        "modelo": ["Exponencial", "Lognormal"],
        "KS_stat": [ks_exp.statistic, ks_ln.statistic],
        "p_value": [ks_exp.pvalue, ks_ln.pvalue],
    }
)
resultados


In [ ]:
xs = np.linspace(0, np.percentile(tiempos, 99.5), 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tiempos, bins=40, density=True, alpha=0.45, label="datos")
ax.plot(xs, stats.expon(loc=loc_exp, scale=scale_exp).pdf(xs), label="Exponencial")
ax.plot(xs, stats.lognorm(s=shape_ln, loc=loc_ln, scale=scale_ln).pdf(xs), label="Lognormal")
ax.set_title("Ajuste de distribuciones continuas")
ax.set_xlabel("tiempo")
ax.set_ylabel("densidad")
ax.legend()
plt.show()


## 3. Probabilidad de riesgo

Con el modelo seleccionado calculamos una probabilidad útil para decisión.


In [ ]:
limite = 12
modelo_lognormal = stats.lognorm(s=shape_ln, loc=loc_ln, scale=scale_ln)
p_supera_limite = 1 - modelo_lognormal.cdf(limite)

print(f"P(tiempo > {limite}) = {p_supera_limite:.4f}")


## 4. Teorema del límite central

La población original será exponencial, claramente asimétrica. Observamos la distribución de promedios.


### Lectura matemática

- **Resultado usado:** $ar{X}pprox N(\mu,\sigma^2/n)$ para $n$ grande.
- **Parámetro estimado:** media poblacional mediante promedios muestrales.
- **Supuesto que puede fallar:** dependencia fuerte o varianza infinita.
- **Diagnóstico:** distribución simulada de promedios para distintos tamaños muestrales.


In [ ]:
poblacion = stats.expon(scale=10)
tamanos = [2, 5, 30, 100]
repeticiones = 8_000

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.ravel()

for ax, n in zip(axes, tamanos):
    muestras = poblacion.rvs(size=(repeticiones, n), random_state=rng)
    promedios = muestras.mean(axis=1)
    mu = poblacion.mean()
    sigma_media = poblacion.std() / np.sqrt(n)
    xs = np.linspace(promedios.min(), promedios.max(), 300)

    ax.hist(promedios, bins=45, density=True, alpha=0.55, label="promedios")
    ax.plot(xs, stats.norm(mu, sigma_media).pdf(xs), color="crimson", label="aprox normal")
    ax.set_title(f"n = {n}")
    ax.legend()

plt.tight_layout()
plt.show()


## 5. Plantilla mínima para trabajo final


In [ ]:
plantilla = {
    "pregunta": "¿Cuál es la probabilidad de superar un umbral operativo?",
    "variable_aleatoria": "Tiempo de atención por solicitud",
    "soporte": "x >= 0",
    "modelo_candidato": "Lognormal",
    "parametros": {"shape": float(shape_ln), "scale": float(scale_ln)},
    "probabilidad_clave": {f"P(X>{limite})": float(p_supera_limite)},
    "validacion": "Comparar histograma, CDF empírica, KS y sentido del negocio",
}
plantilla


## 6. Modelos continuos con tráfico de Lima

Este bloque reutiliza los CSV de `resultados_futuro/`: duración en tráfico como variable continua positiva y comparación de modelos continuos.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "resultados_futuro").exists():
    print("No se encontró data_sources/resultados_futuro/. Se mantiene la sección sintética.")
else:
    archivos = sorted((DATA_DIR / "resultados_futuro").glob("*.csv"))
    trafico = pd.concat([pd.read_csv(a, parse_dates=["timestamp_local"]) for a in archivos], ignore_index=True)
    duracion = trafico["duracion_en_trafico_min"].dropna()
    duracion = duracion[duracion > 0]

    exp_params = stats.expon.fit(duracion, floc=0)
    gamma_params = stats.gamma.fit(duracion, floc=0)
    lognorm_params = stats.lognorm.fit(duracion, floc=0)

    comparacion = pd.DataFrame(
        {
            "modelo": ["Exponencial", "Gamma", "Lognormal"],
            "KS_stat": [
                stats.kstest(duracion, "expon", args=exp_params).statistic,
                stats.kstest(duracion, "gamma", args=gamma_params).statistic,
                stats.kstest(duracion, "lognorm", args=lognorm_params).statistic,
            ],
        }
    ).sort_values("KS_stat")
    display(comparacion)


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "resultados_futuro").exists():
    n = 12
    rep = 5_000
    muestras = rng.choice(duracion.to_numpy(), size=(rep, n), replace=True)
    promedios = muestras.mean(axis=1)
    print(f"Media empírica de duración: {duracion.mean():.2f} min")
    print(f"SE aproximado para promedio de {n} mediciones: {duracion.std(ddof=1) / np.sqrt(n):.2f} min")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(promedios, bins=35, density=True, alpha=0.6)
    ax.set_title("Bootstrap/TLC: promedios de duración en tráfico")
    ax.set_xlabel("promedio de duración")
    plt.show()


## 7. Funciones de pérdida para demanda: MSE, Poisson, Gamma y cuantiles

Este bloque recupera `modelos_demanda.ipynb 3` y `modelos_variables_continuas.ipynb 3`. La idea de clase es conectar el supuesto probabilístico con la función objetivo:

- MSE aproxima media condicional.
- MAE/cuantil 0.5 aproxima mediana condicional.
- Poisson loss es natural para conteos positivos.
- Gamma loss es útil para variables positivas asimétricas.


### Lectura matemática

- **Modelo implícito:** cada pérdida apunta a un funcional distinto de $Y\mid X$.
- **Parámetros estimados:** media condicional, mediana/cuantil o tasa positiva.
- **Supuesto que puede fallar:** pérdida no alineada a inventario, cola o costo operativo.
- **Diagnóstico:** RMSE, MAE, bias, error p90 y análisis de residuales por segmento.


In [ ]:
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró sales_data.csv. Se omite el bloque de pérdidas de demanda.")
else:
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    demanda_df = pd.read_csv(DATA_DIR / "sales_data.csv", parse_dates=["Date"])
    demanda_df = demanda_df.sample(n=min(15000, len(demanda_df)), random_state=42).sort_values("Date")
    demanda_df["month"] = demanda_df["Date"].dt.month
    demanda_df["dayofweek"] = demanda_df["Date"].dt.dayofweek

    features = ["Category", "Region", "Inventory Level", "Price", "Discount", "Weather Condition", "Promotion", "Competitor Pricing", "month", "dayofweek"]
    target = "Demand"
    X = demanda_df[features]
    y = demanda_df[target].astype(float)

    train_mask = demanda_df["Date"] < demanda_df["Date"].quantile(0.75)
    X_train, X_test = X.loc[train_mask], X.loc[~train_mask]
    y_train, y_test = y.loc[train_mask], y.loc[~train_mask]

    cat_cols = ["Category", "Region", "Weather Condition"]
    num_cols = [c for c in features if c not in cat_cols]
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    preprocess = ColumnTransformer([("cat", encoder, cat_cols), ("num", "passthrough", num_cols)])

    modelos_loss = {
        "squared_error_media": HistGradientBoostingRegressor(loss="squared_error", max_iter=120, max_depth=6, learning_rate=0.06, random_state=42),
        "poisson_conteo": HistGradientBoostingRegressor(loss="poisson", max_iter=120, max_depth=6, learning_rate=0.06, random_state=42),
        "gamma_positiva": HistGradientBoostingRegressor(loss="gamma", max_iter=120, max_depth=6, learning_rate=0.06, random_state=42),
        "quantile_p50_mediana": GradientBoostingRegressor(loss="quantile", alpha=0.5, n_estimators=120, max_depth=3, learning_rate=0.06, random_state=42),
    }

    metricas_loss = []
    for nombre, modelo in modelos_loss.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", modelo)])
        try:
            pipe.fit(X_train, y_train)
            pred = np.clip(pipe.predict(X_test), 0, None)
            metricas_loss.append(
                {
                    "modelo_loss": nombre,
                    "rmse": mean_squared_error(y_test, pred, squared=False),
                    "mae": mean_absolute_error(y_test, pred),
                    "bias": float(np.mean(pred - y_test)),
                    "p90_error_abs": float(np.quantile(np.abs(pred - y_test), 0.90)),
                }
            )
        except Exception as exc:
            metricas_loss.append({"modelo_loss": nombre, "error": f"{type(exc).__name__}: {exc}"})
    display(pd.DataFrame(metricas_loss))


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "sales_data.csv").exists():
    transforms = {
        "identity": (lambda z: z, lambda z: z),
        "log1p": (np.log1p, np.expm1),
        "sqrt": (np.sqrt, lambda z: z ** 2),
    }
    transform_rows = []
    for nombre, (forward, inverse) in transforms.items():
        pipe = Pipeline(
            [
                ("preprocess", preprocess),
                ("model", HistGradientBoostingRegressor(loss="squared_error", max_iter=100, max_depth=6, learning_rate=0.06, random_state=42)),
            ]
        )
        pipe.fit(X_train, forward(y_train))
        pred = np.clip(inverse(pipe.predict(X_test)), 0, None)
        transform_rows.append(
            {
                "transformacion_target": nombre,
                "rmse": mean_squared_error(y_test, pred, squared=False),
                "mae": mean_absolute_error(y_test, pred),
                "bias": float(np.mean(pred - y_test)),
            }
        )
    display(pd.DataFrame(transform_rows).sort_values("mae"))


## Práctica

Cambia la población base del TLC a una lognormal o gamma. Evalúa desde qué tamaño muestral la aproximación normal resulta razonable.
